In [ ]:
import wrds
import pandas as pd
from influxdb_client_3 import InfluxDBClient3, Point, WriteOptions, write_client_options

In [ ]:
wrds_db = wrds.Connection()

In [ ]:
def success(self, data: str):
    print(f"Successfully wrote batch: {data}")

def error(self, data: str, exception):
    print(f"Failed writing batch: config: {self}, data: {data} due: {exception}")

def retry(self, data: str, exception):
    print(f"Failed retry writing batch: config: {self}, data: {data} retry: {exception}")

# Configure options for batch writing.
write_options = WriteOptions(
    batch_size=10_000,
    flush_interval=10_000,
    jitter_interval=2_000,
    retry_interval=5_000,
    max_retries=5,
    max_retry_delay=30_000,
    exponential_base=2
)

# Create an options dict that sets callbacks and WriteOptions.
wco = write_client_options(
    success_callback=success,
    error_callback=error,
    retry_callback=retry,
    write_options=write_options
)

client = InfluxDBClient3(
    token="apiv3_hoBjjFB7ZYYtYy33a_Ui5VTjsWG848623y8Mlf_a_2gVU86YB8gwVjJVYRJXaTUWnS4desa23-y7DcXa4qxmIg", 
    host="http://localhost:8181", 
    database="jkp_factors", 
    write_client_options=wco
)

In [ ]:
schema = wrds_db.get_table(library="contrib", table="global_factor", obs=0)
schema.columns

In [ ]:
schema_list = schema.columns.tolist()
print(schema_list)

In [ ]:
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import pandas as pd

class JKPDataLoader:
    """
    Modular data loader for JKP (Jensen, Kelly, Pedersen) global characteristics data
    Validated against JKP_Characteristics_README.md and characteristics.csv
    """
    
    # Factor mappings based on JKP documentation and academic validation
    FACTOR_MODELS = {
        'fama_french_5': {
            # Core Fama-French 5 factors - validated against JKP documentation
            'value': ['be_me', 'ni_me', 'ocf_me'],  # Book-to-market (primary), earnings yield, cash flow yield
            'size': ['market_equity'],  # Market equity (primary FF size measure)
            'momentum': ['ret_12_1'],  # 12-1 month momentum (Jegadeesh Titman 1993)
            'profitability': ['gp_at', 'ni_at'],  # Gross profits/assets (Novy-Marx 2013), ROA
            'investment': ['at_gr1', 'oaccruals_at'],  # Asset growth (Cooper et al 2008), Operating accruals (Sloan 1996)
            'market': ['ret_exc', 'beta_60m']  # Market excess returns and beta
        },
        'fama_french_5_extended': {
            # Extended version with more characteristics per factor
            'value': ['be_me', 'ni_me', 'ocf_me', 'sale_me', 'fcf_me'],
            'size': ['market_equity', 'me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'ret_12_7'],
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'market': ['ret_exc', 'beta_60m', 'beta_252d']
        },
        'quality_comprehensive': {
            # Comprehensive quality-focused model
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be', 'ocf_at'],
            'quality_composite': ['f_score', 'qmj', 'qmj_prof', 'qmj_safety'],
            'earnings_quality': ['ni_ar1', 'earnings_variability', 'oaccruals_at'],
            'growth_quality': ['sale_gr1', 'at_gr1', 'qmj_growth'],
            'financial_strength': ['at_be', 'cash_at', 'z_score']
        },
        'trading_frictions': {
            # Trading frictions and liquidity factors
            'size': ['market_equity', 'me'],
            'volatility': ['ivol_capm_252d', 'ivol_ff3_21d', 'rvol_21d'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'beta': ['beta_60m', 'beta_252d', 'betadown_252d'],
            'skewness': ['iskew_capm_21d', 'rskew_21d', 'coskew_21d'],
            'extremes': ['rmax1_21d', 'rmax5_21d', 'rmax5_rvol_21d']
        },
        'academic_research': {
            # Comprehensive academic research model
            'value': ['be_me', 'ni_me', 'fcf_me', 'sale_me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'resff3_12_1'],
            'profitability': ['gp_at', 'ni_at', 'ope_be', 'f_score'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'volatility': ['ivol_capm_252d', 'beta_60m'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'quality': ['qmj', 'qmj_prof', 'qmj_safety'],
            'intangibles': ['rd_me', 'age', 'tangibility']
        },
        'basic': {
            # Simple 4-factor model
            'returns': ['ret_exc'],
            'size': ['market_equity'],
            'value': ['be_me'],
            'momentum': ['ret_12_1']
        }
    }
    
    def __init__(self, wrds_connection):
        self.wrds_db = wrds_connection
        
    def get_available_characteristics(self) -> List[str]:
        """Get all available characteristics from the JKP dataset"""
        schema = self.wrds_db.get_table(library="contrib", table="global_factor", obs=0)
        return schema.columns.tolist()
    
    def validate_characteristics(self, characteristics: List[str]) -> List[str]:
        """Validate that requested characteristics exist in the dataset"""
        available_chars = self.get_available_characteristics()
        valid_chars = []
        invalid_chars = []
        
        for char in characteristics:
            if char in available_chars:
                valid_chars.append(char)
            else:
                invalid_chars.append(char)
        
        if invalid_chars:
            print(f"Warning: Invalid characteristics removed: {invalid_chars}")
            print(f"Available similar characteristics: {[c for c in available_chars if any(inv in c for inv in invalid_chars)]}")
        
        return valid_chars
    
    def load_characteristics_data(self, 
                                factor_model: str = 'fama_french_5',
                                custom_factors: Optional[List[str]] = None,
                                months_back: int = 12,
                                countries: List[str] = ['USA'],
                                filters: Optional[Dict] = None) -> pd.DataFrame:
        """
        Load characteristics data with flexible parameters
        
        Parameters:
        -----------
        factor_model : str
            Predefined factor model ('fama_french_5', 'fama_french_5_extended', 'quality_comprehensive', 
            'trading_frictions', 'academic_research', 'basic')
        custom_factors : List[str], optional
            Custom list of factor names to load
        months_back : int
            Number of months of historical data to load
        countries : List[str]
            List of countries to include
        filters : Dict, optional
            Additional filters for data quality
        """
        
        # Determine which characteristics to load
        if custom_factors:
            characteristics = custom_factors
        elif factor_model in self.FACTOR_MODELS:
            # Flatten all factors in the model
            characteristics = []
            for factor_group in self.FACTOR_MODELS[factor_model].values():
                characteristics.extend(factor_group)
            # Remove duplicates
            characteristics = list(set(characteristics))
        else:
            raise ValueError(f"Unknown factor model: {factor_model}. Available models: {list(self.FACTOR_MODELS.keys())}")
        
        # Validate characteristics exist in dataset
        characteristics = self.validate_characteristics(characteristics)
        
        if not characteristics:
            raise ValueError("No valid characteristics found!")
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=months_back * 30)  # Approximate
        
        # Default filters for data quality (based on JKP recommendations)
        default_filters = {
            'common': 1,        # Common stocks only
            'exch_main': 1,     # Main exchange only
            'primary_sec': 1,   # Primary security only
            'obs_main': 1       # Main observation only
        }
        
        if filters:
            default_filters.update(filters)
        
        # Build the SQL query
        filter_conditions = []
        for key, value in default_filters.items():
            filter_conditions.append(f"{key}={value}")
        
        # Add country filter
        if countries:
            country_list = "', '".join(countries)
            filter_conditions.append(f"excntry IN ('{country_list}')")
        
        # Add date filter
        filter_conditions.append(f"eom >= '{start_date.strftime('%Y-%m-%d')}'")
        
        # Build characteristics string
        char_string = ', '.join(characteristics)
        
        sql_query = f"""
        SELECT id, eom, gvkey, permno, size_grp, {char_string}
        FROM contrib.global_factor
        WHERE {' AND '.join(filter_conditions)}
        ORDER BY eom DESC
        """
        
        print(f"Loading {len(characteristics)} characteristics for last {months_back} months...")
        print(f"Factor model: {factor_model}")
        print(f"Characteristics: {characteristics}")
        
        return self.wrds_db.raw_sql(sql_query)
    
    def get_factor_model_info(self, model_name: str) -> Dict:
        """Get information about a specific factor model"""
        if model_name in self.FACTOR_MODELS:
            return self.FACTOR_MODELS[model_name]
        else:
            return {}
    
    def list_available_models(self) -> List[str]:
        """List all available factor models"""
        return list(self.FACTOR_MODELS.keys())
    
    def print_model_summary(self, model_name: str):
        """Print detailed summary of a factor model"""
        if model_name not in self.FACTOR_MODELS:
            print(f"Model '{model_name}' not found. Available models: {self.list_available_models()}")
            return
        
        model = self.FACTOR_MODELS[model_name]
        print(f"\n{model_name.upper()} FACTOR MODEL")
        print("=" * 50)
        
        total_chars = 0
        for factor_group, characteristics in model.items():
            print(f"\n{factor_group.upper()}:")
            for char in characteristics:
                print(f"  • {char}")
            total_chars += len(characteristics)
        
        print(f"\nTotal characteristics: {total_chars}")
        print(f"Factor groups: {len(model)}")

# Initialize the data loader
data_loader = JKPDataLoader(wrds_db)

print("📊 JKP Data Loader Initialized")
print("=" * 40)

print("\nAvailable factor models:")
for model in data_loader.list_available_models():
    print(f"  📈 {model}")

# Print detailed summary of the Fama-French 5 model
data_loader.print_model_summary('fama_french_5')

# Load validated Fama-French 5-factor model data
print(f"\n🔄 Loading data...")
usa_data = data_loader.load_characteristics_data(
    factor_model='fama_french_5',
    months_back=12,
    countries=['USA']
)

print(f"\n✅ Successfully loaded {len(usa_data)} observations")
print(f"📅 Date range: {usa_data['eom'].min()} to {usa_data['eom'].max()}")
print(f"🏢 Unique companies: {usa_data['gvkey'].nunique()}")
print(f"📊 Data shape: {usa_data.shape}")

usa_data.head()

# Fama-French 5 Factor Model Characteristics

## Value Factor Characteristics
- **be_me**: Book equity to market equity ratio (classic book-to-market)
- **at_me**: Total assets to market equity ratio
- **sale_me**: Sales to market equity ratio
- **ni_me**: Net income to market equity ratio  
- **ocf_me**: Operating cash flow to market equity ratio
- **ebitda_me**: EBITDA to market equity ratio

## CAPM/Market Factor Characteristics  
- **ret_exc**: Excess returns (return minus risk-free rate)
- **beta_60m**: 60-month market beta
- **beta_252d**: 252-day (1-year) market beta

## Size Factor Characteristics
- **me**: Market equity (shares outstanding × price)
- **market_equity**: Total market value of equity
- **sale_me**: Sales-to-market equity (revenue efficiency)

## Momentum Factor Characteristics
- **ret_12_1**: 12-month return skipping most recent month
- **ret_6_1**: 6-month return skipping most recent month  
- **ret_12_7**: 12-month return skipping most recent 7 months
- **ret_9_1**: 9-month return skipping most recent month

## Quality/Profitability Factor Characteristics
- **ni_at**: Net income to total assets (ROA)
- **ni_be**: Net income to book equity (ROE)
- **roe_be_std**: ROE standard deviation (earnings stability)
- **ocf_at**: Operating cash flow to total assets
- **gp_at**: Gross profits to total assets
- **f_score**: Piotroski F-Score (composite quality measure)

## Additional Factor Models Available
- **profitability**: Focus on earnings, cash flow, margins, returns
- **investment**: Asset growth, capex, financing measures  
- **risk**: Volatility, beta, skewness, maximum return measures
- **basic**: Simple 4-factor model (returns, size, value, momentum)

In [ ]:
# InfluxDB Setup and Data Processing
# ==================================
# 
# For detailed InfluxDB Enterprise Home setup instructions, 
# see: InfluxDB_Setup_README.md in the project root directory

import numpy as np

# Improved data processing for InfluxDB
def prepare_data_for_influxdb(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare data for optimal InfluxDB storage
    """
    processed_data = df.copy().set_index("eom")
    
    # Handle missing values appropriately for time series
    processed_data["size_grp"] = processed_data["size_grp"].fillna("unknown")
    processed_data["permno"] = processed_data["permno"].astype("string").fillna("unknown")
    processed_data["gvkey"] = processed_data["gvkey"].fillna("unknown")
    
    # Convert numeric columns to appropriate types
    numeric_columns = processed_data.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        processed_data[col] = pd.to_numeric(processed_data[col], errors='coerce')
    
    # Add metadata columns for better querying
    processed_data['data_vintage'] = datetime.now().strftime('%Y-%m-%d')
    processed_data['factor_model'] = 'fama_french_5'
    
    return processed_data

# Process the data
processed_data = prepare_data_for_influxdb(usa_data)
print(f"Processed data shape: {processed_data.shape}")
processed_data.head()

In [ ]:
class InfluxDBManager:
    """
    Manager class for InfluxDB operations with the JKP factor data
    """
    
    def __init__(self, client):
        self.client = client
    
    def write_characteristics_data(self, df: pd.DataFrame, measurement_name: str = "jkp_characteristics"):
        """
        Write characteristics data to InfluxDB with proper tagging
        """
        try:
            # Define tag columns (categorical data for efficient querying)
            tag_columns = ["size_grp", "gvkey", "permno", "factor_model"]
            
            # Write to InfluxDB
            self.client.write(
                df, 
                data_frame_measurement_name=measurement_name,
                data_frame_tag_columns=tag_columns
            )
            print(f"Successfully wrote {len(df)} records to {measurement_name}")
            return True
            
        except Exception as e:
            print(f"Error writing to InfluxDB: {e}")
            return False
    
    def query_data(self, query: str):
        """
        Query data from InfluxDB
        """
        try:
            result = self.client.query(query)
            return result
        except Exception as e:
            print(f"Error querying InfluxDB: {e}")
            return None
    
    def get_factor_data(self, 
                       gvkeys: List[str] = None, 
                       start_date: str = None, 
                       end_date: str = None,
                       factors: List[str] = None):
        """
        Convenience method to query specific factor data
        """
        
        # Build query conditions
        conditions = []
        if gvkeys:
            gvkey_list = "', '".join(gvkeys)
            conditions.append(f"gvkey IN ('{gvkey_list}')")
        
        if start_date:
            conditions.append(f"time >= '{start_date}'")
        
        if end_date:
            conditions.append(f"time <= '{end_date}'")
        
        where_clause = " AND ".join(conditions) if conditions else "true"
        
        # Build field selection
        field_selection = "*"
        if factors:
            field_selection = ", ".join(factors)
        
        query = f"""
        SELECT {field_selection}
        FROM jkp_characteristics
        WHERE {where_clause}
        ORDER BY time DESC
        """
        
        return self.query_data(query)

# Initialize InfluxDB manager and write data
influx_manager = InfluxDBManager(client)

# Write the processed characteristics data
success = influx_manager.write_characteristics_data(processed_data)

if success:
    print("Data successfully stored in InfluxDB!")
    print("\nSample query - Recent data for a specific GVKEY:")
    
    # Example query to verify data was written
    sample_gvkey = processed_data['gvkey'].iloc[0] if not processed_data.empty else None
    if sample_gvkey and sample_gvkey != 'unknown':
        recent_data = influx_manager.get_factor_data(
            gvkeys=[sample_gvkey],
            start_date="2024-01-01"
        )
        if recent_data is not None:
            print(f"Found data for GVKEY {sample_gvkey}")
else:
    print("Failed to write data to InfluxDB")

In [ ]:
# Data Granularity Analysis: Beyond Monthly EOM Data
# ===================================================

def analyze_data_granularity():
    """
    Analyze available data frequencies in the JKP dataset and WRDS
    """
    print("JKP Global Factor Dataset Analysis:")
    print("=" * 50)
    
    # Check the date frequency in our current data
    if not usa_data.empty:
        dates = pd.to_datetime(usa_data['eom']).sort_values()
        date_diffs = dates.diff().dropna()
        
        print(f"Current dataset date range: {dates.min()} to {dates.max()}")
        print(f"Number of unique dates: {len(dates.unique())}")
        print(f"Most common date difference: {date_diffs.mode().iloc[0] if not date_diffs.empty else 'N/A'}")
        
        # Analyze frequency
        if len(dates) > 1:
            avg_diff = date_diffs.mean()
            if avg_diff.days < 7:
                frequency = "Daily or higher"
            elif avg_diff.days < 14:
                frequency = "Weekly"
            elif avg_diff.days < 32:
                frequency = "Monthly"
            elif avg_diff.days < 100:
                frequency = "Quarterly"
            else:
                frequency = "Annual or lower"
            
            print(f"Detected frequency: {frequency}")
    
    print("\nData Granularity Options in WRDS:")
    print("-" * 40)
    print("1. JKP Global Factor (contrib.global_factor): Monthly EOM")
    print("2. CRSP Daily Stock File: Daily returns and prices")
    print("3. CRSP Monthly Stock File: Monthly returns")
    print("4. Compustat Daily: Daily fundamental ratios (limited)")
    print("5. Compustat Quarterly: Quarterly fundamentals")
    print("6. TAQ (if available): Intraday tick data")
    
    print("\nRecommendations for Factor Modeling:")
    print("-" * 40)
    print("• Monthly EOM (JKP): Best for factor research, comprehensive characteristics")
    print("• Daily returns (CRSP): For portfolio backtesting and risk management")
    print("• Quarterly fundamentals (Compustat): For custom characteristic construction")
    print("• Intraday (TAQ): For microstructure research (usually not needed for factors)")

def check_higher_frequency_data():
    """
    Check if higher frequency data is available
    """
    print("\nChecking for higher frequency data sources...")
    
    # Check CRSP daily data availability
    try:
        crsp_daily_schema = wrds_db.get_table(library="crsp", table="dsf", obs=0)
        print(f"✓ CRSP Daily Stock File available with {len(crsp_daily_schema.columns)} columns")
        print("  Key columns: date, permno, ret, prc, vol, shrout")
    except:
        print("✗ CRSP Daily Stock File not accessible")
    
    # Check monthly CRSP
    try:
        crsp_monthly_schema = wrds_db.get_table(library="crsp", table="msf", obs=0)
        print(f"✓ CRSP Monthly Stock File available with {len(crsp_monthly_schema.columns)} columns")
    except:
        print("✗ CRSP Monthly Stock File not accessible")
    
    # Check Compustat quarterly
    try:
        comp_quarterly_schema = wrds_db.get_table(library="comp", table="fundq", obs=0)
        print(f"✓ Compustat Quarterly available with {len(comp_quarterly_schema.columns)} columns")
    except:
        print("✗ Compustat Quarterly not accessible")

# Run the analysis
analyze_data_granularity()
check_higher_frequency_data()

In [ ]:
# Data Export and Optimization Utilities
# ======================================

class DataExportManager:
    """
    Manager for exporting data in various optimized formats
    """
    
    @staticmethod
    def export_to_parquet(df: pd.DataFrame, filename: str, compression: str = 'snappy'):
        """
        Export to Parquet format (recommended for large datasets)
        Parquet advantages: Columnar storage, compression, fast queries
        """
        try:
            df.to_parquet(f"{filename}.parquet", compression=compression)
            print(f"✓ Exported to {filename}.parquet with {compression} compression")
            return True
        except Exception as e:
            print(f"✗ Error exporting to Parquet: {e}")
            return False
    
    @staticmethod
    def export_to_hdf5(df: pd.DataFrame, filename: str, key: str = 'data'):
        """
        Export to HDF5 format (good for time series)
        HDF5 advantages: Hierarchical structure, fast I/O, compression
        """
        try:
            df.to_hdf(f"{filename}.h5", key=key, mode='w', complevel=9)
            print(f"✓ Exported to {filename}.h5")
            return True
        except Exception as e:
            print(f"✗ Error exporting to HDF5: {e}")
            return False
    
    @staticmethod
    def export_to_feather(df: pd.DataFrame, filename: str):
        """
        Export to Feather format (fastest for pandas)
        Feather advantages: Fastest pandas I/O, preserves data types
        """
        try:
            df.to_feather(f"{filename}.feather")
            print(f"✓ Exported to {filename}.feather")
            return True
        except Exception as e:
            print(f"✗ Error exporting to Feather: {e}")
            return False
    
    @staticmethod
    def compare_file_sizes(base_filename: str):
        """
        Compare file sizes of different formats
        """
        import os
        
        formats = ['.csv', '.parquet', '.h5', '.feather']
        sizes = {}
        
        for fmt in formats:
            filepath = f"{base_filename}{fmt}"
            if os.path.exists(filepath):
                size_mb = os.path.getsize(filepath) / (1024 * 1024)
                sizes[fmt] = size_mb
        
        print("\nFile Size Comparison:")
        print("-" * 30)
        for fmt, size in sorted(sizes.items(), key=lambda x: x[1]):
            print(f"{fmt:10}: {size:.2f} MB")

# Example usage and recommendations
def export_current_data():
    """
    Export current data in multiple formats for comparison
    """
    if not usa_data.empty:
        base_filename = f"jkp_fama_french_5_{datetime.now().strftime('%Y%m%d')}"
        
        # Export in different formats
        exporter = DataExportManager()
        
        print("Exporting data in multiple formats...")
        exporter.export_to_parquet(usa_data, base_filename)
        exporter.export_to_hdf5(usa_data, base_filename)
        exporter.export_to_feather(usa_data, base_filename)
        
        # Also export as CSV for comparison
        usa_data.to_csv(f"{base_filename}.csv", index=False)
        print(f"✓ Exported to {base_filename}.csv")
        
        # Compare sizes
        exporter.compare_file_sizes(base_filename)
        
        print("\nRecommendations:")
        print("• Parquet: Best for long-term storage and cross-platform compatibility")
        print("• Feather: Fastest for pandas-only workflows")
        print("• HDF5: Good for hierarchical data and time series")
        print("• CSV: Human-readable but largest file size")
        print("• InfluxDB: Best for time series queries and real-time analytics")

# Run export comparison if data is available
if 'usa_data' in locals() and not usa_data.empty:
    export_current_data()
else:
    print("No data available for export. Please run the data loading cells first.")

In [ ]:
factor_returns = pd.read_csv("factor-returns.csv")
factor_returns = factor_returns.set_index("date")
factor_returns

In [ ]:
client.write(factor_returns, 
             data_frame_measurement_name="factor_returns", 
             data_frame_tag_columns=["location", "name", "freq","weighting"])

In [ ]:
wrds_db.list_tables(library="comp")

In [ ]:
comp_comp = wrds_db.get_table(library="comp", table="company")
comp_keys = comp_comp.gvkey.unique()

In [ ]:
wrds_db.list_tables(library="crsp")

In [ ]:
comp_crsp_linker = wrds_db.get_table(library="crsp", table="ccmxpf_lnkhist")

In [ ]:
link_keys = comp_crsp_linker.gvkey.unique()

In [ ]:
comp_comp[comp_comp.gvkey=="011271"]

In [ ]:
a = set(comp_keys)
b = set(link_keys)
a-b